In [2]:
!pip install -q transformers datasets peft accelerate
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn
!pip install -q bert-score rouge-score evaluate tqdm requests scipy
print("Packages installed. Restart kernel if prompted.")

Packages installed. Restart kernel if prompted.


In [3]:
import os, json, warnings, time, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import requests, io

from tqdm import tqdm
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict, load_dataset as hf_load
from bert_score import score as bert_score
from evaluate import load as ev_load

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "gpt2"
MAX_LENGTH = 256
BATCH_SIZE = 2
MAX_STEPS  = 80
GRAD_ACCUM = 1
LORA_R     = 8
LORA_ALPHA = 32

print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME} + LoRA (r={LORA_R})")
print(f"Steps  : {MAX_STEPS} per task")

SYSTEM_NAME = "Single-LLM"

Device : cuda
Model  : gpt2 + LoRA (r=8)
Steps  : 80 per task


In [4]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'Workplace']


In [5]:
import os

print(os.path.exists('/content/drive'))
print(os.listdir('/content/drive'))

True
['MyDrive', '.shortcut-targets-by-id', '.Trash-0', '.Encrypted']


In [29]:


import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Show files in Workplace folder
workplace_path = '/content/drive/MyDrive/Workplace'

if os.path.exists(workplace_path):
    files = os.listdir(workplace_path)
    if files:
        print("Files in Workplace:")
        for f in files:
            print(" -", f)
    else:
        print("Workplace folder is empty")
else:
    print("Workplace folder not found, create it in Google Drive first")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in Workplace:
 - .keep
 - labelled_testing_data.csv
 - labelled_training_data.csv


In [30]:
import os
print(os.path.exists("/content/drive"))

True


In [31]:
from google.colab import drive
drive.mount('/content/drive')
# drive.mount('/content/drive', force_remount=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import os
print(os.listdir('/content/drive/MyDrive'))
print(os.listdir('/content/drive/MyDrive/Workplace'))

['Workplace']
['.keep', 'labelled_testing_data.csv', 'labelled_training_data.csv', 'labelled_validation_data.csv']


In [33]:
import os

LOCAL_DATA_DIR = "/content/drive/MyDrive/Workplace"

print("Exists:", os.path.exists(LOCAL_DATA_DIR))
print("Files:", os.listdir(LOCAL_DATA_DIR))

Exists: True
Files: ['.keep', 'labelled_testing_data.csv', 'labelled_training_data.csv', 'labelled_validation_data.csv']


In [34]:
# Dataset loading helpers
LOCAL_DATA_DIR = r"/content/drive/MyDrive/Workplace"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
NROWS = 3000
SYNTHETIC_USED = []

def download_csv(url, nrows=NROWS, label=""):
    try:
        print(f"  Downloading {label}...")
        r = requests.get(url, timeout=50, stream=True)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.content.decode("utf-8", errors="replace")), nrows=nrows)
        print(f"  Downloaded: {len(df):,} rows")
        return df
    except Exception as e:
        print(f"  Failed: {str(e)[:60]}")
        return None

def load_hf(repo_id, split="train", nrows=NROWS, label=""):
    try:
        print(f"  HuggingFace: {repo_id}...")
        ds = hf_load(repo_id, split=split)
        df = ds.to_pandas().head(nrows)
        print(f"  Loaded: {len(df):,} rows")
        return df
    except Exception as e:
        print(f"  HuggingFace failed: {str(e)[:60]}")
        return None

def load_local(fname, nrows=NROWS):
    path = os.path.join(LOCAL_DATA_DIR, fname)
    if os.path.exists(path):
        df = pd.read_csv(path, nrows=nrows)
        print(f"  Local: {fname} ({len(df):,} rows)")
        return df
    return None

def first_valid(*dfs):
    for df in dfs:
        if df is not None and not df.empty:
            return df
    return None

def synthetic_cicids(n=NROWS):
    np.random.seed(SEED)
    lbl = np.random.choice(["BENIGN","DoS","PortScan","BruteForce","Web Attack"],
                            n, p=[0.60,0.15,0.10,0.10,0.05])
    return pd.DataFrame({
        "Flow Duration":np.random.randint(0,10000,n),
        "Total Fwd Packets":np.random.randint(1,100,n),
        "Total Bwd Packets":np.random.randint(0,50,n),
        "Flow Bytes/s":np.random.uniform(0,1e6,n),
        "Flow Packets/s":np.random.uniform(0,10000,n),
        "Fwd Packet Length Max":np.random.randint(0,1500,n),
        "Bwd Packet Length Max":np.random.randint(0,1500,n),
        "FIN Flag Count":np.random.randint(0,2,n),
        "SYN Flag Count":np.random.randint(0,5,n),
        "Label":lbl,
        "Label_Binary":(lbl!="BENIGN").astype(int),
    })

LOG_TPL = ["PROCESS pid={pid} user={user} cmd={cmd} result=success",
           "NETWORK src={src} dst={dst} port={port} proto=TCP flags=SYN",
           "FILE path=/var/log/{f} op={op} size={sz}B",
           "AUTH user={user} method=password result={res}",
           "SYSCALL pid={pid} call={call} args=fd={fd} retval=0"]

def _log():
    t = random.choice(LOG_TPL)
    return t.format(pid=random.randint(1000,9999),
        user=random.choice(["root","admin","svc","user1"]),
        cmd=random.choice(["bash","python3","curl","nmap"]),
        src=f"192.168.{random.randint(0,255)}.{random.randint(1,254)}",
        dst=f"10.0.{random.randint(0,10)}.{random.randint(1,50)}",
        port=random.choice([22,80,443,8080,3306,4444]),
        f=random.choice(["syslog","auth.log","kern.log"]),
        op=random.choice(["read","write","delete"]),sz=random.randint(100,50000),
        res=random.choice(["success","failure"]),
        call=random.choice(["open","write","execve"]),fd=random.randint(0,10))

def synthetic_beth(n=2000):
    random.seed(SEED)
    rows = []
    for _ in range(n):
        s = random.random() < 0.15
        rows.append({"log":_log(),"sus":int(s),
                     "evil":int(s and random.random()<0.3),
                     "processId":random.randint(1000,9999),
                     "userId":random.randint(0,5)})
    return pd.DataFrame(rows)

print(f"Data folder: {os.path.abspath(LOCAL_DATA_DIR)}")

Data folder: /content/drive/MyDrive/Workplace


In [35]:
# Load all datasets

print("="*65); print("LOADING DATASETS"); print("="*65)
SYNTHETIC_USED.clear()

print("\n[1/3] CICIDS 2017/2018 (Sharafaldin et al., 2018)")
cicids_df = first_valid(
    load_local("Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv"),
    load_local("cicids2018.csv"),
    download_csv("https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/"
        "Processed%20Traffic%20Data%20for%20ML%20Algorithms/"
        "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv", label="CICIDS AWS"))
if cicids_df is None:
    print("  WARNING: Using synthetic. Download from kaggle.com/datasets/chethuhn/network-intrusion-dataset")
    cicids_df = synthetic_cicids(); SYNTHETIC_USED.append("CICIDS")
else:
    cicids_df.columns = cicids_df.columns.str.strip()
    lc = [c for c in cicids_df.columns if "label" in c.lower()]
    if lc and "Label" not in cicids_df.columns:
        cicids_df = cicids_df.rename(columns={lc[0]:"Label"})
    if "Label" not in cicids_df.columns: cicids_df["Label"] = "BENIGN"
    cicids_df["Label"] = cicids_df["Label"].astype(str).str.strip()
    cicids_df["Label_Binary"] = (cicids_df["Label"] != "BENIGN").astype(int)
    for col in ["Flow Duration","Total Fwd Packets","Total Bwd Packets","Flow Bytes/s",
                "Flow Packets/s","SYN Flag Count","Fwd Packet Length Max",
                "Bwd Packet Length Max","FIN Flag Count"]:
        if col not in cicids_df.columns: cicids_df[col] = 0
print(f"  Rows: {len(cicids_df):,} | Attack: {cicids_df['Label_Binary'].mean():.1%}")

print("\n[2/3] BETH Dataset (Highnam et al., 2021)")
def load_beth_local():
    found = []
    for fname in ["labelled_training_data.csv","labelled_testing_data.csv","labelled_validation_data.csv"]:
        part = load_local(fname)
        if part is not None: found.append(part)
    if not found: return None
    combined = pd.concat(found, ignore_index=True)
    print(f"  Combined: {len(combined):,} rows")
    return combined

def norm_beth(df):
    df = df.copy(); df.columns = df.columns.str.strip().str.lower()
    if "log" not in df.columns:
        df["log"] = df.apply(lambda r: f"PROCESS pid={r.get('processid',0)} "
                             f"user={r.get('userid',0)} event={r.get('eventid',0)}", axis=1)
    df["sus"]  = df.get("sus",  pd.Series(0,index=df.index)).fillna(0).astype(int)
    df["evil"] = df.get("evil", pd.Series(0,index=df.index)).fillna(0).astype(int)
    return df

beth_raw = first_valid(
    load_beth_local(),
    load_hf("KateHighnam/beth-dataset", split="train", nrows=NROWS, label="BETH"))
if beth_raw is None:
    print("  WARNING: Using synthetic. Download from kaggle.com/datasets/katehighnam/beth-dataset")
    beth_df = synthetic_beth(); SYNTHETIC_USED.append("BETH")
else:
    beth_df = norm_beth(beth_raw).head(NROWS)
print(f"  Rows: {len(beth_df):,} | Suspicious: {beth_df['sus'].sum()}")

print("\n[3/3] Alert Dataset (generated from SOC templates)") # the
SEV_MAP    = {'LOW':0,'MEDIUM':1,'HIGH':2,'CRITICAL':3}
SEV_LABELS = ['LOW','MEDIUM','HIGH','CRITICAL']
ALERT_TEMPLATES = [
    {"alert":"SSH brute-force: {n} logins in 60s from {src}","sev":"HIGH"},
    {"alert":"SQL injection attempt from {src}","sev":"CRITICAL"},
    {"alert":"Port scan from {src}: 1000 ports in 10s","sev":"MEDIUM"},
    {"alert":"Antivirus update failed on {host}","sev":"LOW"},
    {"alert":"Lateral movement: {user} on 5 systems in 2min","sev":"CRITICAL"},
    {"alert":"Ransomware indicators on {host}","sev":"CRITICAL"},
    {"alert":"Phishing email opened by {user}","sev":"HIGH"},
    {"alert":"Outbound 50GB in 30min from {host}","sev":"HIGH"},
    {"alert":"Failed privilege escalation on {host}","sev":"HIGH"},
    {"alert":"Malware beacon from {host} to {src}","sev":"CRITICAL"},
]
RESP_MAP = {
    "CRITICAL": "Immediate isolation. Activate incident response.",
    "HIGH"    : "Urgent investigation within 1 hour. Block source.",
    "MEDIUM"  : "Investigate within 4 hours. Log pattern.",
    "LOW"     : "Schedule next review cycle. No immediate action.",
}
def build_alert_df(n=1200):
    random.seed(SEED); rows = []
    for _ in range(n):
        t = random.choice(ALERT_TEMPLATES)
        alert = t['alert'].format(
            src=f"192.168.{random.randint(1,255)}.{random.randint(1,254)}",
            host=f"WS-{random.randint(1,99):02d}",
            user=random.choice(['jsmith','admin','bknown','svc_acc']),
            n=random.randint(100,1000))
        rows.append({'alert':alert,'sev':t['sev'],'label':SEV_MAP[t['sev']]})
    return pd.DataFrame(rows)

alert_df = build_alert_df(1200)
print(f"  Alert rows: {len(alert_df):,}")
print("  Distribution:", alert_df['sev'].value_counts().to_dict())

print("\n" + "="*65)
if SYNTHETIC_USED:
    print(f"  WARNING - Synthetic data used for: {', '.join(SYNTHETIC_USED)}")
else:
    print("  All datasets loaded from real sources.")
print("="*65)

LOADING DATASETS

[1/3] CICIDS 2017/2018 (Sharafaldin et al., 2018)
  Downloaded: 3,000 rows
  Rows: 3,000 | Attack: 100.0%

[2/3] BETH Dataset (Highnam et al., 2021)
  Local: labelled_training_data.csv (3,000 rows)
  Local: labelled_testing_data.csv (3,000 rows)
  Local: labelled_validation_data.csv (3,000 rows)
  Combined: 9,000 rows
  HuggingFace: KateHighnam/beth-dataset...
  HuggingFace failed: Dataset 'KateHighnam/beth-dataset' doesn't exist on the Hub 
  Rows: 3,000 | Suspicious: 273

[3/3] Alert Dataset (generated from SOC templates)
  Alert rows: 1,200
  Distribution: {'HIGH': 490, 'CRITICAL': 453, 'MEDIUM': 134, 'LOW': 123}

  All datasets loaded from real sources.


In [36]:
# Section 4: Baseline Models for All Three Tasks

from sklearn.feature_extraction.text import TfidfVectorizer

print("Training Task 1 baseline: TF-IDF Extractive Summarizer on BETH logs...")

def make_summary(group):
    sus = group['sus'].sum(); evil = group['evil'].sum(); total = len(group)
    logs = " | ".join(group['log'].tolist())
    if evil > 0:   sev, act = "CRITICAL", f"{evil} malicious events require immediate response."
    elif sus > 0:  sev, act = "WARNING",  f"{sus} suspicious events require investigation."
    else:          sev, act = "INFO",     "All events normal. No action required."
    return f"[{sev}] Window of {total}. Suspicious:{sus}. Malicious:{evil}. {act}", logs

log_records = []
indices = list(range(0, len(beth_df)-8, 8)); random.shuffle(indices)
for start in indices[:600]:
    group = beth_df.iloc[start:start+8]
    summary, logs = make_summary(group)
    log_records.append({'input':logs, 'output':summary,
                        'sus':group['sus'].sum(), 'evil':group['evil'].sum()})
log_df = pd.DataFrame(log_records)

tr_log, tmp = train_test_split(log_df, test_size=0.20, random_state=SEED)
va_log, te_log = train_test_split(tmp, test_size=0.50, random_state=SEED)

# TF-IDF baseline: score each log line, pick highest scoring as "summary"
def tfidf_summarize(log_window, sus_count, evil_count):
    lines = log_window.split(" | ")
    if len(lines) == 1:
        selected = lines[0]
    else:
        tfidf = TfidfVectorizer(max_features=50)
        try:
            tfidf.fit(lines)
            scores = tfidf.transform(lines).sum(axis=1).A1
            selected = lines[int(np.argmax(scores))]
        except:
            selected = lines[0]
    total = len(lines)
    if evil_count > 0:
        return f"[CRITICAL] Window of {total}. Suspicious:{sus_count}. Malicious:{evil_count}. {evil_count} malicious events require immediate response."
    elif sus_count > 0:
        return f"[WARNING] Window of {total}. Suspicious:{sus_count}. Malicious:0. {sus_count} suspicious events require investigation."
    else:
        return f"[INFO] Window of {total}. Suspicious:0. Malicious:0. All events normal. No action required."

tfidf_preds = []
tfidf_refs  = []
for _, row in te_log.iterrows():
    pred = tfidf_summarize(row['input'], row['sus'], row['evil'])
    tfidf_preds.append(pred)
    tfidf_refs.append(row['output'])

print("Computing TF-IDF baseline BERTScore...")
_, _, F1_tfidf = bert_score(tfidf_preds, tfidf_refs, lang="en", verbose=False)
tfidf_bertscore = round(F1_tfidf.mean().item(), 4)

rouge_metric = ev_load("rouge")
tfidf_rouge  = rouge_metric.compute(predictions=tfidf_preds, references=tfidf_refs)

print(f"  TF-IDF Baseline BERTScore F1 : {tfidf_bertscore:.4f}")
print(f"  TF-IDF Baseline ROUGE-1      : {tfidf_rouge['rouge1']:.4f}")
print(f"  TF-IDF Baseline ROUGE-L      : {tfidf_rouge['rougeL']:.4f}")
BASELINE_T1 = {
    'BERTScore_F1': tfidf_bertscore,
    'ROUGE1': round(tfidf_rouge['rouge1'], 4),
    'ROUGEL': round(tfidf_rouge['rougeL'], 4),
}

# --- Task 2 Baseline: Random Forest (Anomaly Detection) ---
FEATURE_COLS = ["Flow Duration","Total Fwd Packets","Total Bwd Packets",
                "Flow Bytes/s","Flow Packets/s","SYN Flag Count",
                "Fwd Packet Length Max","Bwd Packet Length Max","FIN Flag Count"]

def prep_cicids(df):
    feats = df[FEATURE_COLS].copy()
    return feats.fillna(0).replace([np.inf,-np.inf], 0)

def ensure_balanced(df, label_col="Label_Binary", n_each=500):
    classes = df[label_col].unique()
    if len(classes) >= 2: return df
    np.random.seed(SEED); n = min(n_each, len(df))
    benign = pd.DataFrame({
        "Flow Duration":np.random.randint(100,1000,n),
        "Total Fwd Packets":np.random.randint(1,10,n),
        "Total Bwd Packets":np.random.randint(0,5,n),
        "Flow Bytes/s":np.random.uniform(100,5000,n),
        "Flow Packets/s":np.random.uniform(1,50,n),
        "Fwd Packet Length Max":np.random.randint(40,500,n),
        "Bwd Packet Length Max":np.random.randint(0,400,n),
        "FIN Flag Count":np.zeros(n,dtype=int),
        "SYN Flag Count":np.zeros(n,dtype=int),
        "Label":"BENIGN","Label_Binary":0})
    return pd.concat([df, benign], ignore_index=True)

print("\nTraining Task 2 baseline: Random Forest on CICIDS 2018...")
cicids_b = ensure_balanced(cicids_df)
X_c = prep_cicids(cicids_b); y_c = cicids_b["Label_Binary"].values
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_c, y_c,
    test_size=0.20, random_state=SEED, stratify=y_c)
sc2 = StandardScaler()
X_tr2_sc = sc2.fit_transform(X_tr2); X_te2_sc = sc2.transform(X_te2)
rf2 = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf2.fit(X_tr2_sc, y_tr2)
rf2_preds = rf2.predict(X_te2_sc)

def detection_metrics(yt, yp):
    yt = np.array(yt); yp = np.array(yp)
    TP=int(((yp==1)&(yt==1)).sum()); TN=int(((yp==0)&(yt==0)).sum())
    FP=int(((yp==1)&(yt==0)).sum()); FN=int(((yp==0)&(yt==1)).sum())
    prec=TP/(TP+FP+1e-9); rec=TP/(TP+FN+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return dict(F1=round(f1,4),Precision=round(prec,4),Recall=round(rec,4),
                FPR=round(FP/(FP+TN+1e-9),4),TPR=round(rec,4),
                TP=TP,TN=TN,FP=FP,FN=FN)

rf2_m = detection_metrics(y_te2, rf2_preds)
BASELINE_T2 = {'RandomForest': rf2_m}
rf2_sub_m   = rf2_m
print(f"  RF Baseline Task 2 F1: {rf2_m['F1']:.4f} | FPR: {rf2_m['FPR']:.4f}")

# --- Task 3 Baseline: Random Forest (Alert Triage) ---
print("\nTraining Task 3 baseline: Random Forest on Alert data...")

def encode_alerts(df):
    return pd.DataFrame({
        'has_ssh'    : df['alert'].str.contains('SSH|brute',      case=False).astype(int),
        'has_sql'    : df['alert'].str.contains('SQL|injection',  case=False).astype(int),
        'has_scan'   : df['alert'].str.contains('scan|probe',     case=False).astype(int),
        'has_lateral': df['alert'].str.contains('lateral',        case=False).astype(int),
        'has_malware': df['alert'].str.contains('malware|ransom', case=False).astype(int),
        'has_exfil'  : df['alert'].str.contains('outbound|exfil', case=False).astype(int),
        'has_priv'   : df['alert'].str.contains('privilege',      case=False).astype(int),
        'has_phish'  : df['alert'].str.contains('phishing',       case=False).astype(int),
        'alert_len'  : df['alert'].str.len(),
    })

X_al = encode_alerts(alert_df); y_al = alert_df['label'].values
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X_al, y_al,
    test_size=0.20, random_state=SEED)
rf3 = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf3.fit(X_tr3, y_tr3)
rf3_preds = rf3.predict(X_te3)
rf3_f1 = f1_score(y_te3, rf3_preds, average='weighted', zero_division=0)
BASELINE_T3 = {'RandomForest': {'F1_weighted': round(rf3_f1,4)}}
print(f"  RF Baseline Task 3 F1: {rf3_f1:.4f}")

print("\nAll baseline models trained.")
print(f"  Task 1 baseline (TF-IDF) BERTScore F1 : {BASELINE_T1['BERTScore_F1']:.4f}")
print(f"  Task 2 baseline (RF)     F1 weighted   : {BASELINE_T2['RandomForest']['F1']:.4f}")
print(f"  Task 3 baseline (RF)     F1 weighted   : {BASELINE_T3['RandomForest']['F1_weighted']:.4f}")

Training Task 1 baseline: TF-IDF Extractive Summarizer on BETH logs...
Computing TF-IDF baseline BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  TF-IDF Baseline BERTScore F1 : 1.0000
  TF-IDF Baseline ROUGE-1      : 1.0000
  TF-IDF Baseline ROUGE-L      : 1.0000

Training Task 2 baseline: Random Forest on CICIDS 2018...
  RF Baseline Task 2 F1: 1.0000 | FPR: 0.0000

Training Task 3 baseline: Random Forest on Alert data...
  RF Baseline Task 3 F1: 1.0000

All baseline models trained.
  Task 1 baseline (TF-IDF) BERTScore F1 : 1.0000
  Task 2 baseline (RF)     F1 weighted   : 1.0000
  Task 3 baseline (RF)     F1 weighted   : 1.0000


In [37]:
!pip uninstall -y torchao
!pip install -U peft transformers accelerate

In [38]:
# Load GPT-2 and apply LoRA adapters

print("Loading GPT-2 tokeniser...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading GPT-2 model...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    bias="none", task_type=TaskType.CAUSAL_LM,
    target_modules=["c_attn"],
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Model ready. Trainable: {trainable:,} ({100*trainable/total:.4f}%)")
print("Base weights frozen - LoRA prevents catastrophic forgetting.")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

def make_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir, num_train_epochs=1, max_steps=MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=5e-4, warmup_steps=10, weight_decay=0.01,
        logging_steps=10, eval_strategy="steps",
        eval_steps=MAX_STEPS, save_steps=MAX_STEPS,
        load_best_model_at_end=False, report_to="none",
        use_cpu=(DEVICE=="cpu"),
    )

def tokenize_batch(examples):
    texts = [f"{i} {o}{tokenizer.eos_token}"
             for i,o in zip(examples['input'],examples['output'])]
    tok = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding='max_length')
    tok['labels'] = tok['input_ids'].copy()
    return tok

def generate_text(prompt, max_new=80):
    enc = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=MAX_LENGTH-max_new)
    enc = {k:v.to(model.device) for k,v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    new_tok = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(new_tok, skip_special_tokens=True).strip()

Loading GPT-2 tokeniser...
Loading GPT-2 model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model ready. Trainable: 294,912 (0.2364%)
Base weights frozen - LoRA prevents catastrophic forgetting.


In [39]:
# !pip uninstall -y torchao
# !pip install -U peft transformers accelerate

In [40]:
# Task 1 - Log Summarization
# One model, trained on BETH logs

log_ds_rows = []
indices = list(range(0, len(beth_df)-8, 8)); random.shuffle(indices)
for start in indices[:600]:
    group = beth_df.iloc[start:start+8]
    sus = group['sus'].sum(); evil = group['evil'].sum()
    logs = " | ".join(group['log'].tolist())
    if evil > 0:   sev, act = "CRITICAL", f"{evil} malicious events require immediate response."
    elif sus > 0:  sev, act = "WARNING",  f"{sus} suspicious events require investigation."
    else:          sev, act = "INFO",     "All events normal. No action required."
    summary = f"[{sev}] Window of 8. Suspicious:{sus}. Malicious:{evil}. {act}"
    log_ds_rows.append({'input':f"Summarise these logs:\n{logs}\nSUMMARY:",'output':summary})

log_df2 = pd.DataFrame(log_ds_rows)
tr,tmp = train_test_split(log_df2, test_size=0.20, random_state=SEED)
va,te  = train_test_split(tmp,     test_size=0.50, random_state=SEED)
log_ds = DatasetDict({'train':Dataset.from_pandas(tr.reset_index(drop=True)),
                      'validation':Dataset.from_pandas(va.reset_index(drop=True)),
                      'test':Dataset.from_pandas(te.reset_index(drop=True))})
tok_log = log_ds.map(tokenize_batch, batched=True, remove_columns=['input','output'])

print("SINGLE-LLM: Training Task 1 - Log Summarization (BETH)")
trainer_t1 = Trainer(model=model, args=make_training_args("./single_t1"),
    train_dataset=tok_log['train'], eval_dataset=tok_log['validation'],
    data_collator=data_collator)
t0 = time.time(); trainer_t1.train()
print(f"Task 1 done in {time.time()-t0:.0f}s")

Map:   0%|          | 0/299 [00:00<?, ? examples/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

SINGLE-LLM: Training Task 1 - Log Summarization (BETH)


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
80,0.461356,0.457231


Task 1 done in 18s


In [41]:
# Task 2 - Anomaly Detection
# Same single model continues training on CICIDS data

def build_anomaly_ds(df, n=600):
    s = df.sample(n=min(n,len(df)), random_state=SEED); recs = []
    for _,row in s.iterrows():
        lbl = str(row.get('Label','BENIGN')).strip()
        is_at = int(row.get('Label_Binary', int(lbl!='BENIGN')))
        prompt = (f"Classify this network flow as BENIGN or ATTACK.\n"
                  f"SYN Flags:{row.get('SYN Flag Count',0):.0f} "
                  f"FlowBytes:{row.get('Flow Bytes/s',0):.0f} "
                  f"FwdPkts:{row.get('Total Fwd Packets',0):.0f}\nClassification:")
        ans = f"ATTACK ({lbl}). Abnormal traffic." if is_at else "BENIGN. Normal traffic."
        recs.append({'input':prompt,'output':ans,'label':is_at})
    return pd.DataFrame(recs)

anomaly_df = build_anomaly_ds(cicids_df, 600)
tr2,tmp2 = train_test_split(anomaly_df, test_size=0.20, random_state=SEED)
va2,te2  = train_test_split(tmp2, test_size=0.50, random_state=SEED)
anomaly_ds = DatasetDict({
    'train':Dataset.from_pandas(tr2[['input','output']].reset_index(drop=True)),
    'validation':Dataset.from_pandas(va2[['input','output']].reset_index(drop=True)),
    'test':Dataset.from_pandas(te2[['input','output']].reset_index(drop=True))})
t2_true = te2['label'].tolist()
tok_an = anomaly_ds.map(tokenize_batch, batched=True, remove_columns=['input','output'])

print("SINGLE-LLM: Training Task 2 - Anomaly Detection (CICIDS 2018)")
trainer_t2 = Trainer(model=model, args=make_training_args("./single_t2"),
    train_dataset=tok_an['train'], eval_dataset=tok_an['validation'],
    data_collator=data_collator)
t0 = time.time(); trainer_t2.train()
print(f"Task 2 done in {time.time()-t0:.0f}s")

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

SINGLE-LLM: Training Task 2 - Anomaly Detection (CICIDS 2018)


Step,Training Loss,Validation Loss
80,0.194123,0.114484


Task 2 done in 14s


In [42]:
# Task 3 - Alert Triage
# Same single model continues training on Alert data

triage_recs = []
for _,row in alert_df.iterrows():
    triage_recs.append({
        'input':f"Triage this alert. Assign CRITICAL/HIGH/MEDIUM/LOW.\nALERT: {row['alert']}\nSEVERITY:",
        'output':f"{row['sev']}: {RESP_MAP[row['sev']]}",
        'label':row['label'], 'alert':row['alert']})
triage_df2 = pd.DataFrame(triage_recs)
trT,tmpT = train_test_split(triage_df2, test_size=0.20, random_state=SEED)
vaT,teT  = train_test_split(tmpT, test_size=0.50, random_state=SEED)
triage_ds = DatasetDict({
    'train':Dataset.from_pandas(trT[['input','output']].reset_index(drop=True)),
    'validation':Dataset.from_pandas(vaT[['input','output']].reset_index(drop=True)),
    'test':Dataset.from_pandas(teT[['input','output']].reset_index(drop=True))})
t3_true = teT['label'].tolist()
tok_tri = triage_ds.map(tokenize_batch, batched=True, remove_columns=['input','output'])

print("SINGLE-LLM: Training Task 3 - Alert Triage (Alert dataset)")
trainer_t3 = Trainer(model=model, args=make_training_args("./single_t3"),
    train_dataset=tok_tri['train'], eval_dataset=tok_tri['validation'],
    data_collator=data_collator)
t0 = time.time(); trainer_t3.train()
print(f"Task 3 done in {time.time()-t0:.0f}s")

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

SINGLE-LLM: Training Task 3 - Alert Triage (Alert dataset)


Step,Training Loss,Validation Loss
80,1.461124,1.289181


Task 3 done in 12s


In [43]:
# Evaluate Single-LLM on all three tasks + measure latency

import psutil
model.eval()
TASK_SCORES = {}

# Task 1 evaluation
t1_predictions, t1_references = [], []
t1_times = []
for i in tqdm(range(len(log_ds['test'])), desc="Task 1 eval"):
    s = log_ds['test'][i]
    t0 = time.time()
    p = generate_text(s['input'])
    t1_times.append((time.time()-t0)*1000)
    t1_predictions.append(p if p else "No output.")
    t1_references.append(s['output'])

P,R,F1_bs = bert_score(t1_predictions, t1_references, lang="en", verbose=False)
bs_f1 = F1_bs.mean().item()
rouge_out = ev_load("rouge").compute(predictions=t1_predictions, references=t1_references)
try:
    bleu_out = ev_load("bleu").compute(
        predictions=[p.split() for p in t1_predictions],
        references=[[r.split()] for r in t1_references])
    bleu_score = bleu_out.get('bleu', 0.0)
except: bleu_score = 0.0
TASK_SCORES[0] = round(bs_f1, 4)
lat_t1 = round(sum(t1_times)/len(t1_times), 1)
print(f"Task 1 BERTScore F1: {TASK_SCORES[0]:.4f} | Latency: {lat_t1} ms")

# Task 2 evaluation
t2_preds_text = []
t2_times = []
for i in tqdm(range(len(anomaly_ds['test'])), desc="Task 2 eval"):
    s = anomaly_ds['test'][i]
    t0 = time.time()
    p = generate_text(s['input'])
    t2_times.append((time.time()-t0)*1000)
    t2_preds_text.append(p if p else "BENIGN")

t2_llm_preds = [1 if 'ATTACK' in p.upper() else 0 for p in t2_preds_text]
t2_llm_m = detection_metrics(t2_true, t2_llm_preds)
TASK_SCORES[1] = t2_llm_m['F1']
lat_t2 = round(sum(t2_times)/len(t2_times), 1)
print(f"Task 2 F1: {TASK_SCORES[1]:.4f} | FPR: {t2_llm_m['FPR']:.4f} | Latency: {lat_t2} ms")

# Task 3 evaluation
def parse_sev(text):
    t = text.upper()
    for sev in ['CRITICAL','HIGH','MEDIUM','LOW']:
        if sev in t: return SEV_MAP[sev]
    return SEV_MAP['LOW']

t3_preds_text = []
t3_times = []
for i in tqdm(range(len(triage_ds['test'])), desc="Task 3 eval"):
    s = triage_ds['test'][i]
    t0 = time.time()
    p = generate_text(s['input'])
    t3_times.append((time.time()-t0)*1000)
    t3_preds_text.append(p if p else "LOW")

t3_llm_preds = [parse_sev(p) for p in t3_preds_text]
t3_f1_llm = f1_score(t3_true, t3_llm_preds, average='weighted', zero_division=0)
t3_f1_mac  = f1_score(t3_true, t3_llm_preds, average='macro',    zero_division=0)
TASK_SCORES[2] = round(t3_f1_llm, 4)
lat_t3 = round(sum(t3_times)/len(t3_times), 1)
mem_mb = round(psutil.Process(os.getpid()).memory_info().rss/(1024*1024), 1)
print(f"Task 3 F1: {TASK_SCORES[2]:.4f} | Latency: {lat_t3} ms")

# Classification reports
print("\nClassification Report - Task 2 (LLM):")
print(classification_report(t2_true, t2_llm_preds, labels=[0,1],
      target_names=["BENIGN","ATTACK"], zero_division=0))
print("Classification Report - Task 3 (LLM):")
print(classification_report(t3_true, t3_llm_preds,
      target_names=SEV_LABELS, zero_division=0))

Task 1 eval: 100%|██████████| 38/38 [00:47<00:00,  1.25s/it]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Task 1 BERTScore F1: 0.8365 | Latency: 1251.0 ms


Task 2 eval: 100%|██████████| 60/60 [01:02<00:00,  1.04s/it]


Task 2 F1: 1.0000 | FPR: 0.0000 | Latency: 1034.1 ms


Task 3 eval: 100%|██████████| 120/120 [02:02<00:00,  1.02s/it]

Task 3 F1: 0.4619 | Latency: 1022.4 ms

Classification Report - Task 2 (LLM):
              precision    recall  f1-score   support

      BENIGN       0.00      0.00      0.00         0
      ATTACK       1.00      1.00      1.00        60

    accuracy                           1.00        60
   macro avg       0.50      0.50      0.50        60
weighted avg       1.00      1.00      1.00        60

Classification Report - Task 3 (LLM):
              precision    recall  f1-score   support

         LOW       0.00      0.00      0.00        14
      MEDIUM       0.00      0.00      0.00        13
        HIGH       0.54      0.75      0.63        57
    CRITICAL       0.88      0.39      0.54        36

    accuracy                           0.47       120
   macro avg       0.35      0.29      0.29       120
weighted avg       0.52      0.47      0.46       120



In [44]:
# Print final Single-LLM results summary

print("="*65)
print("SINGLE-LLM RESULTS SUMMARY")
print("="*65)
print(f"  {'Task':<35} {'LLM Score':>10} {'RF Baseline':>12}")
print(f"  {'-'*60}")
print(f"  {'Task 1 BETH      (BERTScore F1)':<35} {TASK_SCORES.get(0,0):>10.4f} {BASELINE_T1['BERTScore_F1']:>12.4f}")
print(f"  {'Task 2 CICIDS    (F1 weighted)':<35} {TASK_SCORES.get(1,0):>10.4f} {BASELINE_T2['RandomForest']['F1']:>12.4f}")
print(f"  {'Task 3 Alerts    (F1 weighted)':<35} {TASK_SCORES.get(2,0):>10.4f} {BASELINE_T3['RandomForest']['F1_weighted']:>12.4f}")
print(f"\n  Avg Latency: {round((lat_t1+lat_t2+lat_t3)/3,1)} ms | Memory: {mem_mb} MB")
print(f"  Models used: 1 (multi-task sequential training)")
print("="*65)

SINGLE-LLM RESULTS SUMMARY
  Task                                 LLM Score  RF Baseline
  ------------------------------------------------------------
  Task 1 BETH      (BERTScore F1)         0.8365       1.0000
  Task 2 CICIDS    (F1 weighted)          1.0000       1.0000
  Task 3 Alerts    (F1 weighted)          0.4619       1.0000

  Avg Latency: 1102.5 ms | Memory: 1859.5 MB
  Models used: 1 (multi-task sequential training)


In [45]:
# Save results to JSON so comparison notebook can load them

import json as json_lib

results = {
    "system"    : SYSTEM_NAME,
    "task1"     : {"BERTScore_F1": TASK_SCORES.get(0,0),
                   "ROUGE1": rouge_out.get('rouge1',0),
                   "ROUGEL": rouge_out.get('rougeL',0),
                   "BLEU": bleu_score},
    "task2"     : t2_llm_m,
    "task3"     : {"F1_weighted": t3_f1_llm, "F1_macro": t3_f1_mac},
    "latency_ms": {
        "task1": round(lat_t1, 1),
        "task2": round(lat_t2, 1),
        "task3": round(lat_t3, 1),
    },
    "memory_mb" : mem_mb,
    "baselines" : {
        "task1_tfidf" : BASELINE_T1,
        "task2_rf"    : BASELINE_T2,
        "task3_rf"    : BASELINE_T3,
    },
}

fname = f"results_{SYSTEM_NAME.lower().replace(' ','_').replace('-','_')}.json"
with open(fname, "w") as f:
    json_lib.dump(results, f, indent=2)
print(f"Results saved to: {fname}")

Results saved to: results_single_llm.json
